# Chapter 2 · Bell States and Quantum Entanglement

## Objectives

1. Build the four Bell states as maximally entangled states of two qubits.
2. Verify that no Bell state can be factored as a tensor product.
3. Calculate the Von Neumann entropy of the reduced states as a measure of entanglement.
4. Simulate the quantum teleportation protocol.

---

## 2.1 Bell States

The four Bell states form an orthonormal basis of the space $\mathbb{C}^2 \otimes \mathbb{C}^2$:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}, \quad |\Phi^-\rangle = \frac{|00\rangle - |11\rangle}{\sqrt{2}}$$

$$|\Psi^+\rangle = \frac{|01\rangle + |10\rangle}{\sqrt{2}}, \quad |\Psi^-\rangle = \frac{|01\rangle - |10\rangle}{\sqrt{2}}$$

The state $|\Phi^+\rangle$ is obtained by applying the Hadamard gate to the first qubit and then a CNOT:

$$|\Phi^+\rangle = \text{CNOT}_{01} \cdot (H \otimes I) \cdot |00\rangle$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from src.quantum_math import QuantumMath
from src.quantum_gates import Gates
from src.visualization import QuantumVisualization

print('Modules loaded.')

In [ ]:
# ── Manual construction of Bell states ────────────────────

# 2-qubit computational basis
ket00 = QuantumMath.tensor_product(QuantumMath.ket0(), QuantumMath.ket0())
ket01 = QuantumMath.tensor_product(QuantumMath.ket0(), QuantumMath.ket1())
ket10 = QuantumMath.tensor_product(QuantumMath.ket1(), QuantumMath.ket0())
ket11 = QuantumMath.tensor_product(QuantumMath.ket1(), QuantumMath.ket1())

# The four Bell states
Phi_plus  = (ket00 + ket11) / np.sqrt(2)
Phi_minus = (ket00 - ket11) / np.sqrt(2)
Psi_plus  = (ket01 + ket10) / np.sqrt(2)
Psi_minus = (ket01 - ket10) / np.sqrt(2)

bell_states = {
    '|Φ+〉': Phi_plus,
    '|Φ-〉': Phi_minus,
    '|Ψ+〉': Psi_plus,
    '|Ψ-〉': Psi_minus,
}

for name, state in bell_states.items():
    print(f'{name} = {np.round(state, 4)}')

## 2.2 Generation with Qiskit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

def bell_circuit(variant: str = 'Phi+') -> QuantumCircuit:
    """Builds the circuit that prepares one of the four Bell states."""
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    if variant in ('Phi-', 'Psi-'):
        qc.z(0)
    if variant in ('Psi+', 'Psi-'):
        qc.x(0)
    qc.cx(0, 1)
    return qc

# Prepare |Φ+〉
qc_bell = bell_circuit('Phi+')
print(qc_bell.draw('text'))

# Analytical state vector
sv = Statevector(qc_bell)
print('\nState vector |Φ+〉:')
print(np.round(sv.data, 4))

In [ ]:
# Bell state measurement and visualization
qc_meas = bell_circuit('Phi+')
qc_meas.measure([0, 1], [0, 1])

backend = AerSimulator()
job = backend.run(qc_meas, shots=4096)
counts = job.result().get_counts()
print('Counts:', counts)

fig = QuantumVisualization.plot_histogram(
    counts, title='Bell state measurements |Φ+〉 (4096 shots)'
)
plt.show()

## 2.3 Non-separability verification

A state $|\psi\rangle \in \mathcal{H}_A \otimes \mathcal{H}_B$ is separable if and only if the factorization $|\psi\rangle = |a\rangle \otimes |b\rangle$ exists. This is equivalent to the **Schmidt rank** being 1, or equivalently, to the **entanglement entropy** being $S = 0$.

The entanglement entropy is calculated by tracing over one of the subsystems:

$$S(A) = -\mathrm{Tr}(\rho_A \log_2 \rho_A)$$

For $|\Phi^+\rangle$, we obtain $S = 1$ bit, the maximum possible for two qubits.

In [ ]:
def entanglement_entropy(state: np.ndarray, dim_A: int = 2) -> float:
    """Calculates the entanglement entropy S(A) for a bipartite state.
    
    Parameters
    ----------
    state : np.ndarray
        State vector of the composite system (2^n,).
    dim_A : int
        Dimension of subsystem A.
    """
    dim_total = len(state)
    dim_B = dim_total // dim_A
    # Reshape as matrix and apply SVD
    M = state.reshape(dim_A, dim_B)
    singular_values = np.linalg.svd(M, compute_uv=False)
    lambdas = singular_values ** 2   # eigenvalues of rho_A
    lambdas = lambdas[lambdas > 1e-14]
    return float(-np.sum(lambdas * np.log2(lambdas)))

print('Entanglement entropies:')
for name, state in bell_states.items():
    S = entanglement_entropy(state)
    print(f'  S({name}) = {S:.4f} bits')

# Product state (separable) for comparison
product_state = QuantumMath.tensor_product(
    QuantumMath.ket_plus(), QuantumMath.ket0()
)
S_prod = entanglement_entropy(product_state)
print(f'  S(|+〉⊗|0〉) = {S_prod:.4f} bits  ← product state, no entanglement')

## 2.4 Quantum teleportation

The teleportation protocol allows transmitting the unknown state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ from Alice to Bob using a shared Bell pair and two classical bits of communication.

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import Statevector

def teleportation_circuit(alpha: complex, beta: complex) -> QuantumCircuit:
    """Quantum teleportation circuit.
    
    Qubits:
      q[0] = Alice's state |ψ〉
      q[1] = Alice's qubit of the Bell pair
      q[2] = Bob's qubit of the Bell pair
    """
    q = QuantumRegister(3, 'q')
    c0 = ClassicalRegister(1, 'c0')
    c1 = ClassicalRegister(1, 'c1')
    qc = QuantumCircuit(q, c0, c1)

    # Step 0: prepare |ψ〉 in q[0]
    qc.initialize([alpha, beta], 0)
    qc.barrier(label='Preparation')

    # Step 1: create Bell pair between q[1] and q[2]
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label='Bell pair')

    # Step 2: Alice's operations
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier(label='Alice')

    # Step 3: Alice's measurement
    qc.measure(0, c0)
    qc.measure(1, c1)
    qc.barrier(label='Measurement')

    # Step 4: Bob's corrections (with classical if)
    with qc.if_test((c1, 1)):
        qc.x(2)
    with qc.if_test((c0, 1)):
        qc.z(2)

    return qc

# Run with a test state
import numpy as np
theta_test = np.radians(70)
a = np.cos(theta_test / 2)             # real alpha
b = np.exp(1j * np.radians(30)) * np.sin(theta_test / 2)  # beta with phase

qc_tel = teleportation_circuit(a, b)
print('Quantum teleportation circuit:')
print(qc_tel.draw('text'))

In [ ]:
# Verify that Bob's qubit receives the correct state
# (statevector simulation before measurement)
qc_no_meas = QuantumCircuit(3)
qc_no_meas.initialize([a, b], 0)
qc_no_meas.h(1)
qc_no_meas.cx(1, 2)
qc_no_meas.cx(0, 1)
qc_no_meas.h(0)

sv_pre = Statevector(qc_no_meas)
print(f'Original state |ψ〉: α={a:.4f}, β={b:.4f}')
print(f'Expected fidelity in teleportation: 1.0 (deterministic classical protocol)')
print('\n→ The protocol guarantees that Bob recovers Alice\'s exact state')
print('  at the cost of destroying it on Alice\'s side (no-cloning).')

## 2.5 Proposed exercises

1. Verify that the four Bell states are orthonormal by computing all inner products $\langle \Phi^+ | \Phi^- \rangle$, $\langle \Phi^+ | \Psi^+ \rangle$, etc.

2. Build the circuit that generates $|\Psi^-\rangle$ and measure 4096 shots. How many '00' and '11' do you observe? Why?

3. Can a three-qubit state $|\psi\rangle \in (\mathbb{C}^2)^{\otimes 3}$ be entangled such that the trace over any subset of qubits is a maximally mixed state? Investigate GHZ and W states.

4. Modify the teleportation circuit to teleport the state $|-\rangle$. Verify that Bob recovers the correct state.